# Verify CTC dataset structure

This notebook verifies the raw-image and annotation structure of the three datasets selected for the Ultrack reproduction:

- `Fluo-N2DL-HeLa`
- `Fluo-N2DH-GOWT1`
- `Fluo-C3DL-MDA231`

Run the notebook on the cluster, where the datasets are already stored. It reads file names and lightweight TIFF metadata; it does not copy data or load complete sequences into memory.

CTC directory roles:

- `01/` and `02/`: raw microscopy images used as model input.
- `01_GT/` and `02_GT/`: ground-truth segmentation and tracking annotations used only for evaluation.
- `01_ST/` and `02_ST/`: silver-truth segmentations used only as references.
- `01_ERR_SEG/` and `02_ERR_SEG/`: intentionally imperfect segmentations supplied for challenge experiments; these are not ground truth and are not raw-image inputs.


## 1. Imports and data location

Set `CTC_ROOT` in the shell before starting Jupyter. For example:

```bash
export CTC_ROOT=/path/to/shared/ctc
jupyter lab --no-browser
```

The active Python environment needs `pandas` and `tifffile`. Both are available in the Ultrack environment used for this project.


In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
from tifffile import TiffFile


In [ ]:
ctc_root_value = os.environ.get("CTC_ROOT")
if not ctc_root_value:
    raise RuntimeError(
        "Set the CTC_ROOT environment variable before starting Jupyter."
    )

CTC_ROOT = Path(ctc_root_value).expanduser()
DATASETS = {
    "Fluo-N2DL-HeLa": 2,
    "Fluo-N2DH-GOWT1": 2,
    "Fluo-C3DL-MDA231": 3,
}
SEQUENCES = ("01", "02")

if not CTC_ROOT.is_dir():
    raise FileNotFoundError(f"CTC root does not exist: {CTC_ROOT}")

print(f"Reading datasets from: {CTC_ROOT}")


## 2. Helper functions

These helpers inspect file names and TIFF metadata. They do not read full image arrays.


In [ ]:
FRAME_NUMBER = re.compile(r"(\d+)$")


def frame_number(path):
    """Return the trailing frame number from a CTC TIFF file name."""
    match = FRAME_NUMBER.search(path.stem)
    if match is None:
        raise ValueError(f"No frame number found in {path.name}")
    return int(match.group(1))


def matching_files(directory, pattern):
    """Return naturally ordered files matching a CTC name pattern."""
    if not directory.is_dir():
        return []
    return sorted(directory.glob(pattern), key=frame_number)


def frame_range(files):
    """Return a compact first-to-last file-name range."""
    if not files:
        return None
    if len(files) == 1:
        return files[0].name
    return f"{files[0].name} to {files[-1].name}"


def tiff_metadata(path):
    """Read shape and dtype without loading a complete TIFF array."""
    with TiffFile(path) as tiff_file:
        series = tiff_file.series[0]
        return tuple(series.shape), str(series.dtype), len(tiff_file.pages)


## 3. Check the expected directories

A standard CTC training dataset has two raw sequences, matching ground-truth folders, and usually silver-truth segmentation folders. This table makes missing directories visible rather than assuming they exist.


In [ ]:
directory_rows = []

for dataset in DATASETS:
    dataset_root = CTC_ROOT / dataset
    for sequence in SEQUENCES:
        directory_rows.append(
            {
                "dataset": dataset,
                "sequence": sequence,
                "dataset root": dataset_root.is_dir(),
                "raw": (dataset_root / sequence).is_dir(),
                "GT/SEG": (dataset_root / f"{sequence}_GT/SEG").is_dir(),
                "GT/TRA": (dataset_root / f"{sequence}_GT/TRA").is_dir(),
                "ST/SEG": (dataset_root / f"{sequence}_ST/SEG").is_dir(),
                "ERR_SEG": (dataset_root / f"{sequence}_ERR_SEG").is_dir(),
            }
        )

directory_summary = pd.DataFrame(directory_rows)
directory_summary


## 4. Count files and record frame ranges

Ground-truth segmentation may cover fewer frames than the raw sequence. A lower `GT segmentation` count is therefore something to record, not automatically an error.


In [ ]:
file_groups = {
    "raw": ("{sequence}", "t*.tif"),
    "GT segmentation": ("{sequence}_GT/SEG", "man_seg*.tif"),
    "GT tracking": ("{sequence}_GT/TRA", "man_track*.tif"),
    "ST segmentation": ("{sequence}_ST/SEG", "man_seg*.tif"),
}
file_rows = []
group_files = {}

for dataset in DATASETS:
    dataset_root = CTC_ROOT / dataset
    for sequence in SEQUENCES:
        for group, (folder_template, pattern) in file_groups.items():
            folder = dataset_root / folder_template.format(sequence=sequence)
            files = matching_files(folder, pattern)
            group_files[(dataset, sequence, group)] = files
            file_rows.append(
                {
                    "dataset": dataset,
                    "sequence": sequence,
                    "file group": group,
                    "count": len(files),
                    "file range": frame_range(files),
                }
            )

file_summary = pd.DataFrame(file_rows)
file_summary


## 5. Inspect one TIFF from each file group

For 2D datasets, a raw TIFF should usually have shape `(y, x)`. For `Fluo-C3DL-MDA231`, each raw TIFF should usually have shape `(z, y, x)`. Annotation masks should have the same spatial shape as the corresponding raw images.


In [ ]:
metadata_rows = []

for dataset, expected_dimensions in DATASETS.items():
    for sequence in SEQUENCES:
        raw_files = group_files[(dataset, sequence, "raw")]
        raw_shape = None
        if raw_files:
            raw_shape, _, _ = tiff_metadata(raw_files[0])

        for group in file_groups:
            files = group_files[(dataset, sequence, group)]
            if not files:
                continue
            shape, dtype, page_count = tiff_metadata(files[0])
            metadata_rows.append(
                {
                    "dataset": dataset,
                    "sequence": sequence,
                    "file group": group,
                    "sample file": files[0].name,
                    "shape": shape,
                    "dtype": dtype,
                    "TIFF pages": page_count,
                    "shape matches raw": shape == raw_shape,
                    "expected raw dimensions": (
                        len(shape) == expected_dimensions
                        if group == "raw"
                        else None
                    ),
                }
            )

metadata_summary = pd.DataFrame(metadata_rows)
metadata_summary


## 6. Compare annotation frame numbers with raw frames

Every annotation TIFF should correspond to an existing raw frame. This check does not require every raw frame to have a segmentation annotation.


In [ ]:
coverage_rows = []

for dataset in DATASETS:
    for sequence in SEQUENCES:
        raw_numbers = {
            frame_number(path)
            for path in group_files[(dataset, sequence, "raw")]
        }
        for group in ("GT segmentation", "GT tracking", "ST segmentation"):
            annotation_numbers = {
                frame_number(path)
                for path in group_files[(dataset, sequence, group)]
            }
            missing_raw_frames = sorted(annotation_numbers - raw_numbers)
            coverage_rows.append(
                {
                    "dataset": dataset,
                    "sequence": sequence,
                    "annotation": group,
                    "annotated frames": len(annotation_numbers),
                    "annotations without raw frame": missing_raw_frames,
                }
            )

coverage_summary = pd.DataFrame(coverage_rows)
coverage_summary


## 7. Verify lineage files

Each `GT/TRA/man_track.txt` row has four integers: cell label, first frame, last frame, and parent label. A parent value of `0` indicates no annotated parent.


In [ ]:
lineage_rows = []

for dataset in DATASETS:
    dataset_root = CTC_ROOT / dataset
    for sequence in SEQUENCES:
        lineage_path = dataset_root / f"{sequence}_GT/TRA/man_track.txt"
        lines = []
        if lineage_path.is_file():
            lines = [
                line.strip()
                for line in lineage_path.read_text().splitlines()
                if line.strip()
            ]
        valid_rows = all(len(line.split()) == 4 for line in lines)
        lineage_rows.append(
            {
                "dataset": dataset,
                "sequence": sequence,
                "lineage file exists": lineage_path.is_file(),
                "lineage records": len(lines),
                "four fields per row": valid_rows if lines else None,
                "first record": lines[0] if lines else None,
            }
        )

lineage_summary = pd.DataFrame(lineage_rows)
lineage_summary


## 8. Record conclusions

After running all cells, record the following in the Week 3 working log:

1. raw-image count, file range, shape, and dtype for each sequence;
2. counts for `GT/SEG`, `GT/TRA`, and `ST/SEG`;
3. whether every annotation has a matching raw frame;
4. whether `man_track.txt` exists and has four fields per row; and
5. any missing or unexpected folders.

Do not use `GT`, `ST`, or `ERR_SEG` images as segmentation input for the Week 3 raw-image workflow. Before committing this notebook, clear its outputs so that file paths and large tables are not stored in Git.
